# 5 — A five-day adjoint of the real global ocean

This notebook computes one number: how the ocean's temperature and salinity after **five
simulated days** would change if you turned the eddy-diffusivity knob. Not by running the
model twice and subtracting — by running it once forwards and once backwards, which gives
the exact derivative at the cost of about three forward runs regardless of how many
parameters you differentiate with respect to.

It runs on the real configuration: the global ocean at 1 degree, real 1958 weather, real
sea ice. All the data is the public release, so this notebook is reproducible outside
DKRZ.

**Two things have to be switched on for this to work at all**, and the bulk of the
notebook is about why. Run it naively and the answer is either meaningless noise or
literally `NaN`.


## What you need

About 11 GB of data and a GPU. The download is one command:

```bash
python scripts/fetch_data.py --dest ~/fesom-data
eval "$(python scripts/fetch_data.py --dest ~/fesom-data --print-env)"
```

That sets `FESOM_MESH_DIR`, `FESOM_IC_DIR`, `FESOM_JRA_DIR` and the rest, which is all this
notebook reads. The record is [doi:10.5281/zenodo.21324319](https://doi.org/10.5281/zenodo.21324319)
and carries 1958 only.

On an A100 the five-day forward takes about three minutes and the backward about seven.
If you have less, set `N_DAYS = 1` below — everything in the notebook still works, and the
reverse pass costs the same memory either way, which is the point of the section on
chunking.


In [ ]:
import os, time
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt

import fesom_jax                       # importing turns on float64
from fesom_jax import ale, ssh, surface_forcing
from fesom_jax.adjoint_stab import AdjointConfig, cotangent_norm
from fesom_jax.gm import GMConfig
from fesom_jax.ice import IceConfig
from fesom_jax.adjoint import scan_steps
from fesom_jax.mesh import load_mesh
from fesom_jax.params import Params
from fesom_jax.calibrate import build_params
from fesom_jax.phc_ic import phc_initial_state
from fesom_jax.step import step
from fesom_jax.tke import TkeConfig

N_DAYS   = 5          # set to 1 on smaller hardware
DT       = 1800.0     # seconds
STEPS_PER_DAY = int(86400 / DT)

MESH_DIR = os.environ['FESOM_MESH_DIR']    # from fetch_data.py --print-env
IC_DIR   = os.environ['FESOM_IC_DIR']
print('device  ', jax.devices()[0])
print('window  ', N_DAYS, 'days =', N_DAYS * STEPS_PER_DAY, 'timesteps')


## The model

The standard realistic setup: variable layer thicknesses, the turbulence-closure vertical
mixing, the eddy parameterization, and prognostic sea ice.


In [ ]:
mesh  = load_mesh(MESH_DIR)
state = phc_initial_state(mesh, IC_DIR)      # observed winter temperature and salinity
op    = ssh.build_ssh_operator(mesh, dt=DT)
cf    = surface_forcing.build_surface_forcing(mesh, 1958, sst_ic=np.asarray(state.T[:, 0]))
dates = surface_forcing.dates_for_steps(1958, DT, N_DAYS * STEPS_PER_DAY)

print(f'{mesh.nod2D:,} surface points, {mesh.nl} levels')
print('starting from 1958-01-01, the initial state in the public release')


## Why the naive answer is wrong

Two components of this model are hostile to reverse-mode differentiation, and neither is a
bug. Both were found by freezing one term at a time and watching what happened to the
growth of the reverse-pass field — the full measurements are in `docs/ADJOINT_HORIZON.md`.

**The turbulence closure.** Its diffusivities depend on the ocean state through thresholds,
clips and reciprocals of the stratification. Differentiating that dependence makes the
reverse pass grow by a factor of 1.21 to 1.55 per timestep — a hundred million over two
days — and the resulting gradient flips sign between runs that are supposed to be
identical. Using the diffusivities the closure computes but *omitting their dependence on
the ocean state* brings the growth down to 1.0070 per step, and three repeat runs then
agree to six significant figures. This is not a trick invented here: it is what the ECCO
ocean state estimation project does, and it is how they run a stable adjoint over a
26-year integration.

**The sea-ice momentum solver.** It relaxes towards a viscous-plastic balance over 120
subcycles per timestep, and its reverse pass amplifies by about 1.18 per subcycle — 4e8 per
model step. That overflows 64-bit floating point inside the first simulated day, so the
exact ice adjoint returns `NaN`. Freezing each piece of an ice step in turn shows the
momentum solver is solely responsible; thermodynamics, advection, the concentration
cut-off and the ocean-to-ice coupling all differentiate fine. Adjoint sea-ice models are
rare for exactly this reason, and every published one either discards the rheology,
reduces it to free drift, simplifies it, or damps the adjoint artificially.

So: two switches. Both are backward-only. `stop_gradient` is the identity in the forward
direction, so **the trajectory you simulate is bit-for-bit the validated model** either
way — only the gradient changes.


In [ ]:
adj = AdjointConfig(freeze_mixing=True)                     # ocean: freeze the mixing coefficients
ice = IceConfig(whichEVP=1, adjoint_mode='rheology')  # ice: freeze only the momentum solve

gm, tke, ale_cfg = GMConfig(), TkeConfig(), ale.AleConfig()
kgm0 = float(Params.defaults().k_gm)

print('ocean :', adj.describe())
print('ice   :', ice.adjoint_mode, '-- thermodynamics, advection, cut-off and the ocean')
print('         coupling stay differentiated; only the ice velocity is dark')


A note on what the ice setting costs you, since it is easy to over- or under-sell. Against
freezing the ice completely, keeping everything except the momentum solve changes this
gradient by about 0.2% over a five-day window. That is well above the numerical noise
floor, so the ice pathway is genuinely contributing — and it is contributing little. Do not
expect sea ice to dominate a temperature-and-salinity gradient at this window length. It
may matter more over months, or for a loss weighted towards the poles.


## The quantity we differentiate

The parameter is a single multiplier on the eddy diffusivity — one number, applied
globally. The loss is the area-weighted mean temperature of the ocean surface after
`N_DAYS`. Both are deliberately simple so the notebook is about the machinery.


In [ ]:
def run_window(state0, m, forcings):
    """Advance the model, with the eddy diffusivity scaled by m."""
    p = build_params({'k_gm': kgm0 * m, 'redi_kmax': kgm0 * m})

    def one_step(s, sf):
        return step(s, mesh, op, None, p, dt=DT, is_first_step=False,
                    step_forcing=sf, forcing_static=cf.static,
                    ice_cfg=ice, gm_cfg=gm, tke_cfg=tke, ale_cfg=ale_cfg,
                    adjoint_cfg=adj, remat_blocks=True)

    # scan_steps derives sqrt(48) = 7 itself; this used to pass segments=7 by hand.
    return scan_steps(one_step, state0, forcings, rematerialize=True)

wet = jnp.asarray(mesh.node_layer_mask[:, 0])

def mean_sst(s):
    return jnp.sum(jnp.where(wet, s.T[:, 0], 0.0)) / jnp.sum(wet)


## Fitting the reverse pass in memory

Reverse-mode differentiation has to keep, or recompute, intermediate values from the
forward pass. Build the whole five-day window as one computation and it will not fit on an
80 GB card — that is what stopped this being measured for a long time.

It does not have to. Run the forward one day at a time and park each day-boundary state in
**ordinary CPU memory**, then walk the days backwards, differentiating one at a time and
carrying only the accumulated sensitivity on the GPU. Peak GPU memory becomes that of a
single day no matter how long the window is; what grows instead is host memory, at about
2 GB per day. **The gradient is exact** — this is bookkeeping about where intermediates
live, not an approximation.

It also hands us a free diagnostic: the size of the sensitivity field at every day
boundary, which is how you tell a healthy adjoint from a diverging one.


In [ ]:
fwd = jax.jit(run_window)

@jax.jit
def bwd(state0, m, forcings, cotangent):
    _, vjp = jax.vjp(lambda s, mm: run_window(s, mm, forcings), state0, m)
    return vjp(cotangent)

m0 = jnp.asarray(1.0, jnp.float64)
chunks = [cf.stack(dates[d*STEPS_PER_DAY:(d+1)*STEPS_PER_DAY]) for d in range(N_DAYS)]

t0 = time.time(); s = state; boundaries = [jax.device_get(s)]
for d in range(N_DAYS):
    s = fwd(s, m0, chunks[d])
    boundaries.append(jax.device_get(s))
    print(f'  day {d+1}: mean SST = {float(mean_sst(s)):.6f} degC', flush=True)
print(f'forward: {time.time()-t0:.0f} s, {sum(a.nbytes for a in jax.tree.leaves(boundaries))/1e9:.1f} GB on the host')


In [ ]:
# seed the reverse pass with the derivative of the loss w.r.t. the final state
loss_val, vjp_loss = jax.vjp(mean_sst, s)
cot = vjp_loss(1.0)[0]

t0 = time.time(); trace = [float(cotangent_norm(cot))]; dLdm = jnp.asarray(0.0, jnp.float64)
for d in reversed(range(N_DAYS)):
    cot, gm_d = bwd(jax.device_put(boundaries[d]), m0, chunks[d], cot)
    dLdm = dLdm + gm_d
    trace.append(float(cotangent_norm(cot)))
    print(f'  back through day {d+1}: |sensitivity| = {trace[-1]:.4e}', flush=True)

print(f'\nbackward: {time.time()-t0:.0f} s')
print(f'\nd(mean SST)/d(eddy-diffusivity multiplier) = {float(dLdm):+.6e} degC per unit')


## Reading the diagnostic

The interesting number is not the gradient, it is how the sensitivity field grew as we
walked backwards. Convert it to a growth rate per timestep. Below about 1.01 you can afford
a window of weeks. At 1.3 you cannot afford days.


In [ ]:
growth = [(trace[i+1]/trace[i])**(1/STEPS_PER_DAY) for i in range(len(trace)-1)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.semilogy(range(N_DAYS, -1, -1), trace, 'o-')
a1.set_xlabel('days before the loss'); a1.set_ylabel('|sensitivity|')
a1.set_title('how the reverse pass grows'); a1.grid(alpha=.3); a1.invert_xaxis()
a2.plot(range(1, N_DAYS+1), growth, 's-')
a2.axhline(1.0, color='k', lw=.8)
a2.set_xlabel('day'); a2.set_ylabel('growth per timestep')
a2.set_title('the number that decides the horizon'); a2.grid(alpha=.3)
plt.tight_layout()

print(f'median growth: {np.median(growth):.4f} per timestep '
      f'=> x{np.median(growth)**(N_DAYS*STEPS_PER_DAY):.3g} over the whole window')


With both switches on this comes out near 1.0004 per timestep, which is a factor of about
1.1 over five days and 1.4 over ten. Without them it is 1.3 per timestep, which is 1e27
over five days — the gradient is then pure amplified round-off, and it changes sign between
identical runs.

If you want to see that for yourself, rerun with `adj = AdjointConfig()` and
`ice = IceConfig(whichEVP=1, adjoint_mode='frozen')`. Do not use `adjoint_mode='exact'`
unless you want to watch it return `NaN`.


## Sanity check against finite differences

Worth doing once, with a caveat that matters. The model is not bit-reproducible on a GPU —
the unstructured-mesh operators sum contributions in whatever order the threads finish — so
the loss wobbles in its last few digits between runs. A finite difference divides by a small
number and amplifies exactly that wobble. Use a *large* step, and treat the agreement as an
order-of-magnitude and sign check rather than a precision test.


In [ ]:
h = 1e-2
def loss_at(m):
    st = state
    for d in range(N_DAYS):
        st = fwd(st, jnp.asarray(m, jnp.float64), chunks[d])
    return float(mean_sst(st))

fd = (loss_at(1.0 + h) - loss_at(1.0 - h)) / (2 * h)
print(f'adjoint          {float(dLdm):+.6e}')
print(f'finite difference {fd:+.6e}   (h = {h})')
print(f'ratio            {float(dLdm)/fd:.3f}')


## What to take away

The gradient of a realistic ocean model over useful timescales is available, and it costs
about three forward runs no matter how many parameters you want derivatives for. But it is
not available by default, and the reason is worth internalising: **the terms that make a
model robust in the forward direction are often the ones that make it fragile in reverse.**
Thresholds, clips, reciprocals of small quantities and subcycled relaxation solvers all
stabilise the simulation and destabilise its derivative.

The fix is not to remove them from the model. It is to stop differentiating *through* them
while still using what they compute — which leaves the simulation untouched and changes
only the gradient. Both switches here do exactly that, and both are what the ocean state
estimation community settled on decades ago.

Two habits are worth copying. Watch the growth rate of the reverse pass, not just the final
number — it tells you whether you are in the regime where the answer means anything. And
compute any gradient at least twice before believing it: on a GPU it need not be the same
number twice, and if it is not, a single sample is not a measurement.

`docs/ADJOINT_HORIZON.md` has the measurements behind every claim here, and
[`04_adjoint_stability.ipynb`](04_adjoint_stability.ipynb) walks through the individual
switches on a small mesh that runs in seconds.
